In [123]:
import pandas as pd

import os
path2data = './raw_data/NL to LTL Synthetic Dataset'
data = os.listdir(path2data)
data = [x for x in data if (  ("train" in x and 'unrestricted' in x and "~" not in x) )]
data

['unrestricted_train_dataset-50.csv', 'unrestricted_train_dataset-140.csv']

In [124]:
import spot
def simplify_formula(formula_str):

    f = spot.formula(formula_str)
    return f


def extract_atomic_propositions(formula_str):
    f = spot.formula(formula_str)
    aps = spot.atomic_prop_collect(f)
    
    mapping = []
    for i, atom in enumerate(aps):
        mapping.append(str(atom))
        

    return mapping


In [125]:
lista = []
for elem in data:
    df = pd.read_csv(path2data + '/' + elem)[['ltl','en']]
    df['source'] = str(elem)
    lista.append( df )
result = pd.concat(lista, ignore_index=True)
result

,ltl,en,source
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv
...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv


In [126]:
result = result.rename(columns={'ltl': 'original LTL', 'en': 'original NL'})

result

,original LTL,original NL,source
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv
...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv


In [127]:

result['Spot LTL'] = result['original LTL'].apply(lambda x: str(simplify_formula(x)))
result['APs'] = result['original LTL'].apply(lambda x: extract_atomic_propositions(x))
result

,original LTL,original NL,source,Spot LTL,APs
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv,G!(brake_is_pressed | house_is_open | train_is...,"[brake_is_pressed, house_is_open, train_is_cro..."
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv,G!(bar_is_up | car_starts | semaphore_is_red),"[car_starts, semaphore_is_red, bar_is_up]"
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv,G!(escalator_speeds_up & house_is_built & trai...,"[escalator_speeds_up, train_derails, house_is_..."
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv,G!(bar_is_closing & constructor_instantiate_ob...,"[constructor_instantiate_objects, bar_is_closi..."
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv,G!(manager_collect_claims & table_has_been_mov...,"[table_has_been_moved, table_is_brown, manager..."
...,...,...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv,G(elevator_rises -> G(sensor_captures_data -> ...,"[train_is_crossing, sensor_captures_data, elev..."
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv,G(train_derails -> G(escalator_moves -> Feleva...,"[train_derails, escalator_moves, elevator_is_b..."
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv,G(brake_is_pressed -> G(constructor_instantiat...,"[brake_is_pressed, constructor_instantiate_obj..."
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv,G(elevator_is_blocked -> G(sensor_captures_dat...,"[sensor_captures_data, elevator_is_blocked, tr..."


In [128]:
import re
def swap_row_propositions(row, prop_col, text_col):
    propositions = row[prop_col]
    text = row[text_col]
    
    if not isinstance(propositions, list) or not isinstance(text, str):
        return text

    # 1. Create mapping for THIS row: {'car starts': 'car_starts'}
    mapping = {p.replace('_', ' '): p for p in propositions}
    
    # 2. Build a regex pattern from these specific keys
    # Sort by length descending to match longest phrases first
    sorted_keys = sorted(mapping.keys(), key=len, reverse=True)
    pattern = re.compile(r'\b(' + '|'.join(map(re.escape, sorted_keys)) + r')\b')
    
    # 3. Replace matches in the text using the mapping
    return pattern.sub(lambda m: mapping[m.group(0)], text)

# Apply the function across the rows
result['ap_NL'] = result.apply(
    lambda row: swap_row_propositions(row, 'APs', 'original NL'), 
    axis=1
)

In [129]:
result

,original LTL,original NL,source,Spot LTL,APs,ap_NL
0,G(!( brake_is_pressed | house_is_open | train_...,under no circumstances either the brake is pre...,unrestricted_train_dataset-50.csv,G!(brake_is_pressed | house_is_open | train_is...,"[brake_is_pressed, house_is_open, train_is_cro...",under no circumstances either the brake_is_pre...
1,G(!( car_starts | semaphore_is_red | bar_is_up )),"under no condition either a car starts, a sema...",unrestricted_train_dataset-50.csv,G!(bar_is_up | car_starts | semaphore_is_red),"[car_starts, semaphore_is_red, bar_is_up]","under no condition either a car_starts, a sema..."
2,G(!( escalator_speeds_up & train_derails & hou...,"under no condition the escalator speeds up, th...",unrestricted_train_dataset-50.csv,G!(escalator_speeds_up & house_is_built & trai...,"[escalator_speeds_up, train_derails, house_is_...","under no condition the escalator_speeds_up, th..."
3,G(!( constructor_instantiate_objects & bar_is_...,it is never the case that a constructor instan...,unrestricted_train_dataset-50.csv,G!(bar_is_closing & constructor_instantiate_ob...,"[constructor_instantiate_objects, bar_is_closi...",it is never the case that a constructor_instan...
4,G(!( table_has_been_moved & table_is_brown & m...,it never happens that the table has been moved...,unrestricted_train_dataset-50.csv,G!(manager_collect_claims & table_has_been_mov...,"[table_has_been_moved, table_is_brown, manager...",it never happens that the table_has_been_moved...
...,...,...,...,...,...,...
189995,G(( elevator_rises ) -> G(( sensor_captures_da...,each time the elevator rises then always when ...,unrestricted_train_dataset-140.csv,G(elevator_rises -> G(sensor_captures_data -> ...,"[train_is_crossing, sensor_captures_data, elev...",each time the elevator_rises then always when ...
189996,G(( train_derails ) -> G(( escalator_moves ) -...,when the train derails then it will happen tha...,unrestricted_train_dataset-140.csv,G(train_derails -> G(escalator_moves -> Feleva...,"[train_derails, escalator_moves, elevator_is_b...",when the train_derails then it will happen tha...
189997,G(( brake_is_pressed ) -> G(( constructor_inst...,the brake is pressed involves that whenever th...,unrestricted_train_dataset-140.csv,G(brake_is_pressed -> G(constructor_instantiat...,"[brake_is_pressed, constructor_instantiate_obj...",the brake_is_pressed involves that whenever th...
189998,G(( elevator_is_blocked ) -> G(( sensor_captur...,when the elevator is blocked then if the senso...,unrestricted_train_dataset-140.csv,G(elevator_is_blocked -> G(sensor_captures_dat...,"[sensor_captures_data, elevator_is_blocked, tr...",when the elevator_is_blocked then if the senso...


In [130]:
result.to_csv('./processed_data/NL to LTL Synthetic Dataset/processed_data.csv', index=False, sep=';')

In [ ]:
def english_to_ltl(s):
    s = s.replace('globally', 'G')
    s = s.replace('finally', 'F')
    s = s.replace('next', 'X')
    s = s.replace('or', '|')
    s = s.replace('double_implies', '<->')
    s = s.replace('implies', '->')
    s = s.replace('and', '&')
    s = s.replace('not', '!')
    s = s.replace('negation', '!')
    s = s.replace('implies', '->')
    return s

In [135]:
path2data = './raw_data/VLTL-Bench/new/'
data = os.listdir(path2data)
data = [x for x in data ]
data

['search_and_rescue.jsonl', 'traffic_light.jsonl', 'warehouse.jsonl']

In [137]:
df = {}
for file in data:
    df[file] = pd.read_json(os.path.join(path2data, file), lines=True)

reserved = ['globally', 'finally', 'next', 'or', 'double_implies', 
            'implies', 'and', 'until', 'not', '(', ')']

def wrap_propositions(token_list):
    return [f'"{item}"' if item not in reserved else item for item in token_list]


for file in df:
    df[file]['source'] = file
    df[file]['original NL'] = df[file]['sentence'].str.join(' ')
    df[file]['tl'] = df[file]['tl'].apply(wrap_propositions)
    df[file]['original LTL'] = df[file]['tl'].str.join(' ').replace('""', '"', regex=True)
    df[file]['Spot LTL'] = df[file]['original LTL'].apply(english_to_ltl)

def get_flat_map(nested_dict):
    # We combine all args_canon -> args_ref pairs into one dict
    if not isinstance(nested_dict, dict):
        return {}
    
    flat_map = {}
    for prop in nested_dict.values():
        # Zip canon keys to ref values and update our master map
        flat_map.update(zip(prop.get('args_ref', []), prop.get('args_canon', [])))
    return flat_map

for file in df:
    df[file]['AP_dict'] = df[file]['prop_dict'].apply(lambda x: {x[y]['action_ref']: x[y]['action_canon'] for y in x})
    
    df[file]['AP_misc_dict'] = df[file]['prop_dict'].apply(get_flat_map)
    df[file]['merged_dict'] = df[file].apply(lambda row: row['AP_misc_dict'] | row['AP_dict'], axis=1)

def replace_from_mapping(row, text_column):
    text = str(row[text_column])
    mapping = row['merged_dict']
    
    # Iterate through the dictionary for this specific row
    for old_val, new_val in mapping.items():
        # replace() handles all instances of that key in the string
        text = text.replace(old_val, new_val)
    
    return text

import csv

for file in df:
    # Apply the function to your text column
    # Replace 'my_text_col' with the actual name of your string column
    df[file]['original NL'] = df[file].apply(replace_from_mapping, axis=1, text_column='original NL')
    df[file]['Spot LTL'] = df[file]['Spot LTL'].apply(lambda x: str(simplify_formula(x)))
    df[file]['APs'] = df[file]['Spot LTL'].apply(lambda x: extract_atomic_propositions(x))
    df[file][['source','APs','original NL','original LTL','Spot LTL']].to_csv('./processed_data/VLTL-Bench/new/'+file.split('.')[0]+'.csv', index=False, sep=';',index_label=False, quoting=csv.QUOTE_MINIMAL)
#

In [138]:
path2data = './raw_data/VLTL-Bench/prev/'
data = os.listdir(path2data)
data = [x for x in data ]
data

['GLTL.jsonl', 'navi.jsonl', 'conformal.jsonl', 'cleanup_world.jsonl']

In [ ]:
df = {}
for file in data:
    df[file] = pd.read_json(os.path.join(path2data, file), lines=True)

reserved = ['globally', 'finally', 'next', 'or', 'double_implies', 
            'implies', 'and', 'until', 'not', 'negation', '(', ')']

def wrap_propositions(token_list):
    return [f'"{item}"' if item not in reserved else item for item in token_list]


for file in df:
    df[file]['source'] = file
    df[file]['original NL'] = df[file]['sentence'].str.join(' ')
    df[file]['tl'] = df[file]['tl'].apply(wrap_propositions)
    df[file]['original LTL'] = df[file]['tl'].str.join(' ').replace('""', '"', regex=True)
    df[file]['Spot LTL'] = df[file]['original LTL'].apply(english_to_ltl)

def get_flat_map(nested_dict):
    # We combine all args_canon -> args_ref pairs into one dict
    if not isinstance(nested_dict, dict):
        return {}
    
    flat_map = {}
    for prop in nested_dict.values():
        # Zip canon keys to ref values and update our master map
        flat_map.update(zip(prop.get('args_ref', []), prop.get('args_canon', [])))
    return flat_map

for file in df:
    df[file]['AP_dict'] = df[file]['prop_dict'].apply(lambda x: {x[y]['action_ref']: x[y]['action_canon'] for y in x})
    
    df[file]['AP_misc_dict'] = df[file]['prop_dict'].apply(get_flat_map)
    df[file]['merged_dict'] = df[file].apply(lambda row: row['AP_misc_dict'] | row['AP_dict'], axis=1)

def replace_from_mapping(row, text_column):
    text = str(row[text_column])
    mapping = row['merged_dict']
    
    # Iterate through the dictionary for this specific row
    for old_val, new_val in mapping.items():
        # replace() handles all instances of that key in the string
        text = text.replace(old_val, new_val)
    
    return text

import csv

for file in df:
    # Apply the function to your text column
    # Replace 'my_text_col' with the actual name of your string column
    df[file]['original NL'] = df[file].apply(replace_from_mapping, axis=1, text_column='original NL')
    df[file]['Spot LTL'] = df[file]['Spot LTL'].apply(lambda x: str(simplify_formula(x)))
    df[file]['APs'] = df[file]['Spot LTL'].apply(lambda x: extract_atomic_propositions(x))
    df[file][['source','APs','original NL','original LTL','Spot LTL']].to_csv('./processed_data/VLTL-Bench/prev/'+file.split('.')[0]+'.csv', index=False, sep=';',index_label=False, quoting=csv.QUOTE_MINIMAL)
#

parse_error: 
>>> "negation" ( "let_go(pear)" ) "imply" "venture_to(trash_can)"
               ^
syntax error, unexpected opening parenthesis

>>> "negation" ( "let_go(pear)" ) "imply" "venture_to(trash_can)"
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ignoring trailing garbage



In [142]:
df[file].T[0]['tl']


['"negation"',
 '(',
 '"let_go(pear)"',
 ')',
 '"imply"',
 '"venture_to(trash_can)"']

In [288]:
import json
from pprint import pprint

with open('./raw_data/ConformalNL2LTL/final_calib.json') as json_data:
    d = json.load(json_data)
    json_data.close()

for ind in range(len(d)):
    d[ind]['ltlequ'] = d[ind]['ltlequ'][0][:-1]
    if "U" in d[ind]['ltlequ']:
       d[ind]['ltlequ'] = d[ind]['ltlequ'].replace("U", " U ")
    
    d[ind]['Spot LTL'] = str(simplify_formula(str(d[ind]['ltlequ'])))

d


[{'nlTask': 'Do not enter garage 1 until you take a picture of landmark 5',
  'ltlequ': '!garage_1 U (landmark_5&&photo)',
  'Spot LTL': '!garage_1 U (landmark_5 & photo)'},
 {'nlTask': 'Ensure you never retrieve package 5',
  'ltlequ': '[]!p_package_5',
  'Spot LTL': 'G!p_package_5'},
 {'nlTask': 'Do not go to street 7 while you take package 3 from store 9 to house 1',
  'ltlequ': '[]!street_7&&<>(store_9&&<>(p_package_3&&<>(house_1&&pd)))',
  'Spot LTL': 'G!street_7 & F(store_9 & F(p_package_3 & F(house_1 & pd)))'},
 {'nlTask': 'Start by visiting mall 5 to pick up box 4, and then move it to house 1',
  'ltlequ': '<>(mall_5&&<>(p_box_4&&<>(house_1&&pd)))',
  'Spot LTL': 'F(mall_5 & F(p_box_4 & F(house_1 & pd)))'},
 {'nlTask': 'Do not go to intersection 1 before arriving at street 6, and ensure you take cone 3 from street 2 and place it in intersection 1',
  'ltlequ': '!intersection_1 U street_6&&<>(street_2&&<>(p_cone_3&&<>(intersection_1&&pd)))',
  'Spot LTL': '(!intersection_1 U str

In [295]:
df = pd.json_normalize(d).rename(columns={'ltlequ': 'original LTL', 'nlTask': 'original NL'})
df['APs'] = df['Spot LTL'].apply(lambda x: extract_atomic_propositions(x))
df['ap_NL'] = df.apply(
    lambda row: swap_row_propositions(row, 'APs', 'original NL'), 
    axis=1
)
df['dict_AP']=df['APs'].apply(lambda x: {x.removeprefix('p_').replace('_', ' ') : x for x in x})

def dynamic_replace(row):
    text = row['ap_NL']
    mapping = row['dict_AP']
    
    # Split the string, replace words found in the dict, join back
    # Using mapping.get(word, word) returns the replacement OR the original word
    return " ".join([mapping.get(word, word) for word in text.split()])
df['ap_NL'] = df.apply(dynamic_replace, axis=1)
df.to_csv('./processed_data/ConformalNL2LTL/processed_data.csv', index=False, sep=';')
df

,original NL,original LTL,Spot LTL,APs,ap_NL,dict_AP
0,Do not enter garage 1 until you take a picture...,!garage_1 U (landmark_5&&photo),!garage_1 U (landmark_5 & photo),"[garage_1, landmark_5, photo]",Do not enter garage_1 until you take a picture...,"{'garage 1': 'garage_1', 'landmark 5': 'landma..."
1,Ensure you never retrieve package 5,[]!p_package_5,G!p_package_5,[p_package_5],Ensure you never retrieve package 5,{'package 5': 'p_package_5'}
2,Do not go to street 7 while you take package 3...,[]!street_7&&<>(store_9&&<>(p_package_3&&<>(ho...,G!street_7 & F(store_9 & F(p_package_3 & F(hou...,"[house_1, pd, p_package_3, street_7, store_9]",Do not go to street_7 while you take package 3...,"{'house 1': 'house_1', 'pd': 'pd', 'package 3'..."
3,"Start by visiting mall 5 to pick up box 4, and...",<>(mall_5&&<>(p_box_4&&<>(house_1&&pd))),F(mall_5 & F(p_box_4 & F(house_1 & pd))),"[mall_5, p_box_4, house_1, pd]","Start by visiting mall_5 to pick up box 4, and...","{'mall 5': 'mall_5', 'box 4': 'p_box_4', 'hous..."
4,Do not go to intersection 1 before arriving at...,!intersection_1 U street_6&&<>(street_2&&<>(p_...,(!intersection_1 U street_6) & F(street_2 & F(...,"[pd, intersection_1, street_6, street_2, p_con...",Do not go to intersection_1 before arriving at...,"{'pd': 'pd', 'intersection 1': 'intersection_1..."
...,...,...,...,...,...,...
673,Go to office 3 to pick up document 1 and move ...,<>(office_3&&<>(p_document_1&&<>(street_2&&[]s...,F(office_3 & F(p_document_1 & F(street_2 & Gst...,"[street_2, office_3, p_document_1]",Go to office_3 to pick up document 1 and move ...,"{'street 2': 'street_2', 'office 3': 'office_3..."
674,Stay in locality 3 until you reach office 4,locality_3 U office_4,locality_3 U office_4,"[office_4, locality_3]",Stay in locality_3 until you reach office_4,"{'office 4': 'office_4', 'locality 3': 'locali..."
675,Go to park 8 and never leave,<>[]park_8,FGpark_8,[park_8],Go to park_8 and never leave,{'park 8': 'park_8'}
676,Avoid entering intersection 5 until you reach ...,!intersection_5 U street_4&&<>(house_4&&<>(p_p...,(!intersection_5 U street_4) & F(house_4 & F(p...,"[pd, intersection_5, street_4, house_4, p_pass...",Avoid entering intersection_5 until you reach ...,"{'pd': 'pd', 'intersection 5': 'intersection_5..."


In [290]:
df

,original NL,original LTL,Spot LTL,APs,ap_NL
0,Do not enter garage 1 until you take a picture...,!garage_1 U (landmark_5&&photo),!garage_1 U (landmark_5 & photo),"[garage_1, landmark_5, photo]",Do not enter garage_1 until you take a picture...
1,Ensure you never retrieve package 5,[]!p_package_5,G!p_package_5,[p_package_5],Ensure you never retrieve package 5
2,Do not go to street 7 while you take package 3...,[]!street_7&&<>(store_9&&<>(p_package_3&&<>(ho...,G!street_7 & F(store_9 & F(p_package_3 & F(hou...,"[house_1, pd, p_package_3, street_7, store_9]",Do not go to street_7 while you take package 3...
3,"Start by visiting mall 5 to pick up box 4, and...",<>(mall_5&&<>(p_box_4&&<>(house_1&&pd))),F(mall_5 & F(p_box_4 & F(house_1 & pd))),"[mall_5, p_box_4, house_1, pd]","Start by visiting mall_5 to pick up box 4, and..."
4,Do not go to intersection 1 before arriving at...,!intersection_1 U street_6&&<>(street_2&&<>(p_...,(!intersection_1 U street_6) & F(street_2 & F(...,"[pd, intersection_1, street_6, street_2, p_con...",Do not go to intersection_1 before arriving at...
...,...,...,...,...,...
673,Go to office 3 to pick up document 1 and move ...,<>(office_3&&<>(p_document_1&&<>(street_2&&[]s...,F(office_3 & F(p_document_1 & F(street_2 & Gst...,"[street_2, office_3, p_document_1]",Go to office_3 to pick up document 1 and move ...
674,Stay in locality 3 until you reach office 4,locality_3 U office_4,locality_3 U office_4,"[office_4, locality_3]",Stay in locality_3 until you reach office_4
675,Go to park 8 and never leave,<>[]park_8,FGpark_8,[park_8],Go to park_8 and never leave
676,Avoid entering intersection 5 until you reach ...,!intersection_5 U street_4&&<>(house_4&&<>(p_p...,(!intersection_5 U street_4) & F(house_4 & F(p...,"[pd, intersection_5, street_4, house_4, p_pass...",Avoid entering intersection_5 until you reach ...


In [291]:
d[0]['ltlequ'][0][:-1]

''

In [183]:
str(simplify_formula('<>(mall_5&&<>(p_box_4&&<>(house_1&&pd)))'))

'F(mall_5 & F(p_box_4 & F(house_1 & pd)))'

In [187]:
str(simplify_formula('!garage_1 U (landmark_5&&photo)'))

'!garage_1 U (landmark_5 & photo)'